# @fifi thumbs: oven + embedded `#image` checks

This notebook is for validating thumbnail generation behavior for **`$odj`** and **`$eep`**,
especially whether embedded `#image` references are present in oven renders.

## What this notebook does
- Pulls live source from `aesthetic.computer/api/store-kidlisp`
- Extracts embedded `#...` references from source
- Resolves painting-code mappings
- Builds production oven URLs with fresh cache keys
- Optionally forces live renders and prints response headers (`x-grab-id`, `x-cache`)
- Shows side-by-side animation/still preview links for manual verification


In [ ]:
from __future__ import annotations

import hashlib
import json
import re
import time
import urllib.parse
import urllib.request
from dataclasses import dataclass
from typing import Dict, List, Any

from IPython.display import HTML, Markdown, display


In [ ]:
# --- Configuration ---
PIECES = ['odj', 'eep']
TOKEN_IDS = {'odj': '28', 'eep': '29'}

BASE_AC = 'https://aesthetic.computer'
BASE_OVEN = 'https://oven.aesthetic.computer'
BASE_TZKT = 'https://api.tzkt.io/v1'

# Flip this to True when you want to force fresh generation (can take ~10-40s/piece).
FORCE_GENERATE = False

# Generation parameters aligned with keep pipeline style.
GEN = {
    'width': 512,
    'height': 512,
    'duration': 8000,
    'fps': 10,
    'quality': 70,
    'density': 2,
    'source': 'keep',
    'skipCache': 'true',
}


In [ ]:
def get_json(url: str, timeout: int = 30) -> Dict[str, Any]:
    req = urllib.request.Request(url, headers={'User-Agent': 'ac-fifi-thumbs-notebook/1.0'})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode('utf-8'))


def get_bytes_with_headers(url: str, timeout: int = 120):
    req = urllib.request.Request(url, headers={'User-Agent': 'ac-fifi-thumbs-notebook/1.0'})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        body = resp.read()
        headers = {k.lower(): v for k, v in resp.headers.items()}
        status = getattr(resp, 'status', 200)
        return status, headers, body


def source_sha16(source: str) -> str:
    return hashlib.sha256((source or '').encode('utf-8')).hexdigest()[:16]


def extract_embedded_refs(source: str) -> List[str]:
    # Captures stamp #txr, paste #abc, etc.
    refs = re.findall(r'#([A-Za-z0-9]{3,})', source or '')
    # Keep order but dedupe
    out = []
    seen = set()
    for r in refs:
        if r not in seen:
            out.append(r)
            seen.add(r)
    return out


In [ ]:
# --- Pull live source + piece metadata ---
piece_data = {}
for code in PIECES:
    url = f"{BASE_AC}/api/store-kidlisp?code={urllib.parse.quote(code)}"
    piece_data[code] = get_json(url)

for code in PIECES:
    d = piece_data[code]
    print(f"${code}")
    print("  handle:", d.get('handle'))
    print("  hits  :", d.get('hits'))
    print("  when  :", d.get('when'))
    print("  source:\n" + (d.get('source') or '').strip())
    print()


In [ ]:
# --- Extract embedded #image refs + resolve mapping ---
embedded_map = {}
resolved_map = {}

for code in PIECES:
    src = piece_data[code].get('source') or ''
    refs = extract_embedded_refs(src)
    embedded_map[code] = refs

    resolved = []
    for ref in refs:
        mapping_url = f"{BASE_AC}/api/painting-code?code={urllib.parse.quote(ref)}"
        m = get_json(mapping_url)
        handle = m.get('handle')
        slug = m.get('slug')
        media_url = f"{BASE_AC}/media/paintings/{ref}.png"
        direct_url = f"{BASE_AC}/media/@{handle}/painting/{slug}.png" if handle and slug else None
        resolved.append({
            'ref': ref,
            'mapping': m,
            'media_url': media_url,
            'direct_url': direct_url,
        })
    resolved_map[code] = resolved

for code in PIECES:
    print(f"${code} refs:", embedded_map[code])
    for row in resolved_map[code]:
        print(" ", row['ref'], "->", row['mapping'])
        print("    media:", row['media_url'])
        print("    direct:", row['direct_url'])
    print()


In [ ]:
def build_oven_urls(piece_code: str, source: str, ts: int | None = None) -> Dict[str, str]:
    ts = int(ts or time.time())
    sha16 = source_sha16(source)

    # Use unique cache keys per run so we can force fresh production captures.
    cache_anim = f"rebake-{sha16}-{ts}-{piece_code}"
    cache_still = f"rebake-{sha16}-{ts}-still-{piece_code}"

    webp_q = {
        'duration': str(GEN['duration']),
        'fps': str(GEN['fps']),
        'quality': str(GEN['quality']),
        'density': str(GEN['density']),
        'source': GEN['source'],
        'skipCache': GEN['skipCache'],
        'cacheKey': cache_anim,
    }
    png_q = {
        'density': str(GEN['density']),
        'source': GEN['source'],
        'skipCache': GEN['skipCache'],
        'cacheKey': cache_still,
    }

    piece_path = urllib.parse.quote(f"${piece_code}")
    webp_url = (
        f"{BASE_OVEN}/grab/webp/{GEN['width']}/{GEN['height']}/{piece_path}?"
        + urllib.parse.urlencode(webp_q)
    )
    png_url = (
        f"{BASE_OVEN}/grab/png/{GEN['width']}/{GEN['height']}/{piece_path}?"
        + urllib.parse.urlencode(png_q)
    )

    return {
        'sha16': sha16,
        'cache_anim': cache_anim,
        'cache_still': cache_still,
        'webp_url': webp_url,
        'png_url': png_url,
    }


oven_urls = {
    code: build_oven_urls(code, piece_data[code].get('source') or '')
    for code in PIECES
}

for code in PIECES:
    u = oven_urls[code]
    print(f"${code}")
    print("  sha16     :", u['sha16'])
    print("  webp_url  :", u['webp_url'])
    print("  png_url   :", u['png_url'])
    print()


In [ ]:
# --- Optional: force live production renders now ---
# This will call oven and download the response bytes.

results = {}

if FORCE_GENERATE:
    for code in PIECES:
        results[code] = {}
        for kind in ['webp_url', 'png_url']:
            url = oven_urls[code][kind]
            try:
                status, headers, body = get_bytes_with_headers(url, timeout=150)
                results[code][kind] = {
                    'status': status,
                    'size': len(body),
                    'content-type': headers.get('content-type'),
                    'x-grab-id': headers.get('x-grab-id'),
                    'x-cache': headers.get('x-cache'),
                    'url': url,
                }
            except Exception as e:
                results[code][kind] = {'error': str(e), 'url': url}

    print(json.dumps(results, indent=2))
else:
    print('FORCE_GENERATE is False. Set it to True and rerun this cell to force production grabs.')


In [ ]:
# --- View links and previews ---
rows = []
for code in PIECES:
    refs_html = []
    for r in resolved_map[code]:
        refs_html.append(
            f"<div><code>#{r['ref']}</code> "
            f"→ <a href='{r['media_url']}' target='_blank'>media</a> "
            f"| <a href='{r['direct_url']}' target='_blank'>direct</a></div>"
        )

    webp = oven_urls[code]['webp_url']
    png = oven_urls[code]['png_url']

    rows.append(f"""
    <tr>
      <td><code>${code}</code></td>
      <td>{''.join(refs_html) or '<em>none</em>'}</td>
      <td>
        <a href='{webp}' target='_blank'>webp url</a><br/>
        <img src='{webp}' width='220' style='border:1px solid #ccc; margin-top:6px;' />
      </td>
      <td>
        <a href='{png}' target='_blank'>png url</a><br/>
        <img src='{png}' width='220' style='border:1px solid #ccc; margin-top:6px;' />
      </td>
    </tr>
    """)

html = f"""
<style>
  table.fifi-check {{ border-collapse: collapse; width: 100%; }}
  .fifi-check th, .fifi-check td {{ border: 1px solid #ddd; padding: 8px; vertical-align: top; }}
  .fifi-check th {{ background: #f6f8fa; text-align: left; }}
</style>
<table class='fifi-check'>
  <thead>
    <tr>
      <th>piece</th>
      <th>embedded refs</th>
      <th>animated webp</th>
      <th>still png</th>
    </tr>
  </thead>
  <tbody>
    {''.join(rows)}
  </tbody>
</table>
"""

display(HTML(html))


In [ ]:
# --- On-chain metadata for fifi token IDs (28, 29) ---
contract = 'KT1Q1irsjSZ7EfUN4qHzAB2t7xLBPsAWYwBB'

def ipfs_to_gateway(uri: str) -> str:
    if not uri or not isinstance(uri, str):
        return ''
    if uri.startswith('ipfs://'):
        cid = uri.replace('ipfs://', '').split('/')[0]
        return f'https://ipfs.aesthetic.computer/ipfs/{cid}'
    return uri

chain = {}
for code, token_id in TOKEN_IDS.items():
    url = f"{BASE_TZKT}/tokens?contract={contract}&tokenId={token_id}"
    payload = get_json(url)
    row = payload[0] if payload else {}
    md = row.get('metadata', {})
    chain[code] = {
        'token_id': token_id,
        'name': md.get('name'),
        'thumbnailUri': md.get('thumbnailUri'),
        'artifactUri': md.get('artifactUri'),
        'displayUri': md.get('displayUri'),
        'description': md.get('description'),
        'thumbnailGateway': ipfs_to_gateway(md.get('thumbnailUri')),
    }

print(json.dumps(chain, indent=2))


## Notes

- This notebook intentionally uses live production endpoints.
- Use unique `cacheKey` values when testing so you can confirm fresh oven output.
- If you want end-to-end keep regeneration, run your keep pipeline after validating the URLs here.
